## 3.0 训练资源估算：参数、显存、FLOPs 与训练时间

对应 **Problem 19：Transformer 模型资源估算**，四问依次为：

| | 问题 |
|---|---|
| (a) | 两套配置的参数量与 FP32 模型显存 |
| (b) | `batch_size = 16 / 32 / 64` 时的训练显存估算 |
| (c) | `batch_size = 64` 时的单步训练 FLOPs |
| (d) | `MFU = 0.4`、训练 10,000 步、RTX 3090 Ti 下的训练时间估算 |

下面一个小问对应一个代码块。

In [1]:
# 两套配置：左为 GPT-2 XL 规模，右为 3.2 节实际训练使用的实验配置
GPT2_XL_SCALE = dict(vocab_size=50257, context_length=1024, num_layers=48, 
                     d_model=1600, num_heads=25, d_ff=4288)
OURS = dict(vocab_size=10000, context_length=256, num_layers=4, d_model=512, num_heads=16, d_ff=1344)

# 公共估算参数
FP32_BYTES = 4
GIB = 1024 ** 3 # 10^9
GPU_PEAK_FLOPS = 8.0e13   # RTX 3090 Ti：BF16 输入、FP32 累加的 Tensor Core 稠密矩阵峰值算力
MFU = 0.4                 # Model FLOPs Utilization = 实际算力/峰值算力

In [2]:
# ==================== (a) 参数量与 FP32 模型显存 ====================

def count_params(vocab_size, d_model, num_layers, num_heads, d_ff):
    """假设 RMSNorm 仅含缩放参数、Linear 无 bias、Embedding 与 LM Head 不共享权重。"""
    assert d_model % num_heads == 0

    embedding = vocab_size * d_model
    per_layer = (
        d_model                    # Attention 前 RMSNorm
        + 4 * d_model * d_model    # Q / K / V / O
        + d_model                  # FFN 前 RMSNorm
        + 3 * d_model * d_ff       # SwiGLU：w1 / w2 / w3
    )
    final_norm = d_model
    lm_head = d_model * vocab_size

    return embedding + num_layers * per_layer + final_norm + lm_head


for name, cfg in [("GPT-2 XL 规模", GPT2_XL_SCALE), ("我们的实验配置", OURS)]:
    n_params = count_params(**{k: v for k, v in cfg.items() if k != "context_length"})
    print(f"{name}: 参数量 {n_params / 1e6:.2f} M ｜ FP32 模型显存 {n_params * FP32_BYTES / GIB:.3f} GiB")

GPT-2 XL 规模: 参数量 1640.45 M ｜ FP32 模型显存 6.111 GiB
我们的实验配置: 参数量 22.70 M ｜ FP32 模型显存 0.085 GiB


In [7]:
# ==================== (b) batch_size = 16 / 32 / 64 时的训练显存估算 ====================

# 训练显存 = 参数 + 梯度 + 优化器状态（AdamW 的 m、v 共两份）+ 激活值
# 为简化估算，本题统一按 FP32，即每个数占 4 Bytes。
# 激活值按朴素 Transformer 实现粗略估算；CUDA 工作区、缓存分配等额外开销暂不考虑。

def estimate_memory(vocab_size, context_length, d_model, num_layers, num_heads, d_ff, batch_size):
    """粗略估算训练时主要显存占用，返回参数、梯度、优化器状态和激活值四部分。"""
    assert d_model % num_heads == 0
    S, d, h, d_head = context_length, d_model, num_heads, d_model // num_heads

    # 输入 token 向量序列（Embedding 输出）：(S, d)，相较于各层主要激活较小，本题忽略。
    # 一个样本 / 一条序列在前向传播过程中需要保存、供反向传播使用的主要激活值
    # ---- 每个 Transformer Block ----
    rms_attn = S * d                              # Attention 前 RMSNorm 的输出

    attn = (
        S * 3 * d                                 # Q、K、V 三份
        + h * S * S * 2                           # Attention score + Softmax probability
        + h * S * d_head                          # 各头逐位置的注意力加权输出（拼接前）
        + S * d * 2                               # 多头拼接结果 + 输出投影结果
    )

    rms_ffn = S * d                               # FFN 前 RMSNorm 的输出

    ffn = (
        S * d_ff * 4                              # w1、w3 输出 + SiLU(w1) + 门控结果
        + S * d                                   # w2 输出
    )

    # ---- 模型输出部分 ----
    final_norm = S * d                            # 最后的 RMSNorm
    logits = S * vocab_size                       # LM Head 输出 logits
    loss = S                                      # 每个位置的 loss

    # 单个样本的激活值总量，再统一乘 batch_size
    activation_per_sample = (
        num_layers * (rms_attn + attn + rms_ffn + ffn)
        + final_norm + logits + loss
    )
    activation = batch_size * activation_per_sample

    # 参数、梯度和优化器状态只与模型规模有关，不随 batch_size 改变
    n_params = count_params(vocab_size, d_model, num_layers, num_heads, d_ff)

    return {
        "参数": n_params * FP32_BYTES,
        "梯度": n_params * FP32_BYTES,             # 梯度与参数一一对应，同样大小
        "优化器状态": 2 * n_params * FP32_BYTES,    # AdamW 的一阶矩 m 与二阶矩 v
        "激活值": activation * FP32_BYTES,
    }


for name, cfg in [("GPT-2 XL 规模", GPT2_XL_SCALE), ("我们的实验配置", OURS)]:
    print(f"\n--- {name} ---")
    for batch_size in [16, 32, 64]:
        mem = estimate_memory(batch_size=batch_size, **cfg)
        parts = " ｜ ".join(f"{k} {v / GIB:.2f} GiB" for k, v in mem.items())
        print(f"batch_size={batch_size:2d} → 合计 {sum(mem.values()) / GIB:.2f} GiB ｜ {parts}")


# 会随 batch_size 变化的主要是激活值，并且在这一粗略模型下近似线性增长；
# 参数、梯度和优化器状态只由模型规模决定。

# 使用 “我们的实验配置” 在 3090 Ti 上跑 batch_size=64，显存使用量大概为 7600 MB，约合 7.4 GB。

# 实际训练峰值显存还会受到 Attention 实现、CUDA 上下文、算子工作区、
# CUDA 缓存分配器等因素影响，因此这里主要用于估计数量级。


--- GPT-2 XL 规模 ---
batch_size=16 → 合计 270.05 GiB ｜ 参数 6.11 GiB ｜ 梯度 6.11 GiB ｜ 优化器状态 12.22 GiB ｜ 激活值 245.60 GiB
batch_size=32 → 合计 515.65 GiB ｜ 参数 6.11 GiB ｜ 梯度 6.11 GiB ｜ 优化器状态 12.22 GiB ｜ 激活值 491.21 GiB
batch_size=64 → 合计 1006.86 GiB ｜ 参数 6.11 GiB ｜ 梯度 6.11 GiB ｜ 优化器状态 12.22 GiB ｜ 激活值 982.41 GiB

--- 我们的实验配置 ---
batch_size=16 → 合计 1.61 GiB ｜ 参数 0.08 GiB ｜ 梯度 0.08 GiB ｜ 优化器状态 0.17 GiB ｜ 激活值 1.27 GiB
batch_size=32 → 合计 2.88 GiB ｜ 参数 0.08 GiB ｜ 梯度 0.08 GiB ｜ 优化器状态 0.17 GiB ｜ 激活值 2.54 GiB
batch_size=64 → 合计 5.42 GiB ｜ 参数 0.08 GiB ｜ 梯度 0.08 GiB ｜ 优化器状态 0.17 GiB ｜ 激活值 5.08 GiB


In [8]:
# ==================== (c) batch_size = 64 时的单步训练 FLOPs ====================

def estimate_forward_flops(vocab_size, context_length, d_model, num_layers, num_heads, d_ff):
    """估算单个样本 / 一条序列完成一次前向传播的主要 FLOPs。

    核心事实：
    矩阵乘法 (M,N) @ (N,P) -> (M,P)，约需要 2*M*N*P FLOPs。

    变量约定：
    S = context_length
    d = d_model
    h = num_heads
    d_head = d // h

    

    RMSNorm、Softmax、SiLU 等逐元素操作相比矩阵乘法计算量较小，本题做粗略估算时忽略。
    比如 RMSNorm 会对序列中的每个 token 向量独立求均方根、做可学习参数缩放等，数量级为 O(S * d)，
    远小于 mha 内部的矩阵乘 O(S * d * d)
    """
    assert d_model % num_heads == 0
    S, d, h, d_head = context_length, d_model, num_heads, d_model // num_heads

    # ---- Multi-Head Attention ----
    mha = (
        2 * S * d * (3 * d)                      # QKV 投影：(S,d) @ (d,3d) -> (S,3d)
        + h * 2 * S * d_head * S                 # Q @ K^T：每个头 (S,d_head) @ (d_head,S) -> (S,S)
        + h * 2 * S * S * d_head                 # Attn @ V：每个头 (S,S) @ (S,d_head) -> (S,d_head)
        + 2 * S * d * d                          # 输出投影：(S,d) @ (d,d) -> (S,d)
    )

    # ---- SwiGLU FFN ----
    ffn = (
        2 * S * d * d_ff                         # w1 变换：(S,d) @ (d,d_ff) -> (S,d_ff)
        + 2 * S * d * d_ff                       # w3 变换：(S,d) @ (d,d_ff) -> (S,d_ff)
        + 2 * S * d_ff * d                       # w2 变换：(S,d_ff) @ (d_ff,d) -> (S,d)
    )

    # ---- 所有 Transformer Block ----
    blocks = num_layers * (mha + ffn)

    # ---- LM Head ----
    lm_head = 2 * S * d * vocab_size             # (S,d) @ (d,V) -> (S,V)

    return {
        "MHA": num_layers * mha,
        "FFN": num_layers * ffn,
        "LM Head": lm_head,
        "合计": blocks + lm_head,
    }


BATCH_SIZE = 64

# 反向传播计算量约为前向的 2 倍：
# 因此：
#   单步训练 FLOPs ≈ 前向 + 反向 ≈ 3 × 前向
# AdamW 参数更新主要是逐元素操作，相比矩阵乘法计算量较小，可忽略。

for name, cfg in [("GPT-2 XL 规模", GPT2_XL_SCALE), ("我们的实验配置", OURS)]:
    # estimate_forward_flops() 返回的是单个样本的前向 FLOPs，
    # 这里统一乘 batch_size 得到整个 batch 的计算量。
    forward = estimate_forward_flops(**cfg)["合计"] * BATCH_SIZE
    backward = 2 * forward
    train_step = forward + backward

    print(
        f"{name}: 前向 {forward / 1e12:.2f} TFLOPs ｜ "
        f"反向 {backward / 1e12:.2f} TFLOPs ｜ "
        f"单步训练 {train_step / 1e12:.2f} TFLOPs"
    )

    # K M G T 10^12

GPT-2 XL 规模: 前向 225.07 TFLOPs ｜ 反向 450.15 TFLOPs ｜ 单步训练 675.22 TFLOPs
我们的实验配置: 前向 0.61 TFLOPs ｜ 反向 1.22 TFLOPs ｜ 单步训练 1.83 TFLOPs


In [9]:
# ==================== (d) MFU = 0.4、10,000 步、RTX 3090 Ti 下的训练时间估算 ====================

# RTX 3090 Ti：
# 假设矩阵乘法采用 BF16 输入、FP32 累加，
# Tensor Core 稠密矩阵理论峰值算力约为 80 TFLOP/s。
#
# MFU = 实际有效算力 / 理论峰值算力
# 因此：实际有效算力 = GPU 峰值算力 × MFU

GPU_PEAK_FLOPS = 8.0e13
MFU = 0.4
BATCH_SIZE = 64
NUM_STEPS = 10_000


def estimate_train_time(train_flops_per_step, gpu_peak_flops, mfu, num_steps):
    """根据单步训练 FLOPs、GPU 峰值算力和 MFU 粗略估算训练时间。"""
    effective_flops = gpu_peak_flops * mfu
    seconds_per_step = train_flops_per_step / effective_flops
    total_seconds = seconds_per_step * num_steps
    return seconds_per_step, total_seconds


for name, cfg in [("GPT-2 XL 规模", GPT2_XL_SCALE), ("我们的实验配置", OURS)]:
    # 先计算单个样本的前向 FLOPs，再统一乘 batch_size
    forward = estimate_forward_flops(**cfg)["合计"] * BATCH_SIZE

    # 反向传播约为前向的 2 倍，因此单步训练约为 3 × 前向
    train_flops_per_step = 3 * forward

    sec_per_step, total_sec = estimate_train_time(
        train_flops_per_step, GPU_PEAK_FLOPS, MFU, NUM_STEPS
    )

    print(
        f"{name}: "
        f"单步训练 {train_flops_per_step / 1e12:.2f} TFLOPs ｜ "
        f"每步约 {sec_per_step:.4f} s ｜ "
        f"{NUM_STEPS:,} 步约 {total_sec / 3600:.2f} 小时"
    )


# MFU = 0.4 是题目给定的理论估算值。
# 实际训练中，尤其对于较小模型，GPU 很难长期达到这一利用率，
# 因此真实训练时间可能明显长于上述估算结果。
# 我使用 “我们的实验配置” 在 3090 Ti 上跑 5000 步，大约花了 1700s，约 0.47 小时。
# 10000 -> 0.92 小时
# 实际 MFU = 0.4 / 6 = 0.065

GPT-2 XL 规模: 单步训练 675.22 TFLOPs ｜ 每步约 21.1006 s ｜ 10,000 步约 58.61 小时
我们的实验配置: 单步训练 1.83 TFLOPs ｜ 每步约 0.0572 s ｜ 10,000 步约 0.16 小时
